# 🫀 퀘스트 46 · Q7-I — **리듬 축을 끝까지 짜낸다** (+ Q7-F 미결 3건 닫기)

| | **MedKOS / `notebooks/quest46_q7i_rhythm_bundle.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | `ailab-2026-0057`(Q7-F) · `ailab-2026-0056`(Q7-H) |
| 규약 | **R11-c · R17 · R20 · R22 · R23 · R24** |
| 학습 | **0회**(특징) + **교차적합 로지스틱**(조합 · 개체 내부) |

## 왜 이 실험인가

Q7-F 가 「형태 축」을 리듬 대리로 판정했다. 그런데 **정작 리듬 축은 특징 하나로
방치돼 있었다** — `median(pre) − pre` 뿐이고, `svdb_data5.npz` 의 **`post_rr` 은 한
번도 안 썼다**. Q7-H 의 교훈이 「기준 교정(+0.0228)보다 **특징 추가**(+0.0601)」였으니
이제 그 트릭을 **리듬 축에** 건다.

### 넣을 특징 넷 (전부 라벨 불필요)

| | 정의 | 왜 |
|---|---|---|
| `f1` | `med(pre) − pre` | 기존 · 비교 기준선 |
| `f2` | `1 − pre / med(pre[i−k:i])` (k=8·16) | S 의 정의는 「**이 환자의 평소**보다 이르다」인데 지금 기준은 **레코드 전체 중앙**이다. 심박수가 드리프트하면 틀린 기준. 비율이라 **스케일 프리** |
| `f3` ★ | `1 − (pre+post) / (2·med(pre[i−k:i]))` | **보상성 휴지.** PAC 은 **비보상성**(pre+post < 2·기준) · PVC 는 **완전보상**(≈). **S 와 V 를 가르는 교과서 축**이고 `post_rr` 은 지금까지 안 썼다 |
| `f4` | `CV(pre[i−k:i])` | **국소 불규칙성.** AF 에서는 「이르다」가 무의미하다. 방향 없는 **문맥 변수** — 단독 AUROC 는 0.5 근처가 정상 |
| `f5` | `f2` 의 **직전 비트 값** | **런/이단맥 문맥.** 직전도 일렀나 |

### ★ `f3` 가 이 실험의 핵심이다

`f1`·`f2` 는 **`pre_rr` 의 함수**라 **심박수를 정합하면 정의상 무력화된다**(같은 대역
안에서는 상수). 그런데 `f3` 는 **`post_rr` 을 쓴다** — 정합 후에도 살아남을 수 있는
유일한 리듬 특징이다. Q7-F 가 「무정합 조건의 숫자는 해석 불가」를 확정했으므로,
**정합 조건에서 뭐가 남느냐**가 진짜 질문이다.

## 사전등록

| 관문 | 내용 | 문턱 |
|---|---|---|
| **I1** | **조합이 기존을 이긴다** — 짝지은 `LR(전부) − LR(f1)` | CI 하한 **> 0** ⚠️ **I2b 가 지지일 때만 읽는다** |
| **I2** | **창 음성 대조를 이긴다** — 짝지은 `LR(전부) − STT` | CI 하한 **> 0** |
| **I2b** ★ | **라벨셔플 null 을 이긴다** — 짝지은 `LR(전부) − LR(셔플)` | CI 하한 **> 0** |
| **I3** ★ | **심박수 정합 조건에서 `f3` 가 남는다** — 정합 `f3` 매크로 | CI 하한 **> 0.5** |
| **I4** | **고립 S 층**에서 조합 > `LR(f1)` | CI 하한 **> 0** |
| **I5** | **런 우세 층**에서 조합이 `LR(f1)` 대비 비열등 | CI 하한 **> −0.02** |

**공정 비교 규약**: `f1` 도 **같은 교차적합 로지스틱**에 태운다(`LR(f1)`). 「적합한 조합
vs 적합 안 한 단일」을 비교하면 그 차이가 곧 결론이 된다.

★ **그런데 그것만으로도 부족하다 — 픽스처가 잡았다.** 교차적합 로지스틱은 **특징 수가
많을수록 null 편의가 커진다**(신호 0 합성 코호트에서 `LR(f1)` 0.417 · `LR(전부)` 0.594 가
나왔다). 1개짜리와 6개짜리를 그냥 비교하면 **그 편의 차가 곧 결론**이 된다.
→ **`LR(전부)` 를 라벨 셔플에 태운 `lr_null` 을 같이 재고 관문(I2b)으로 건다.**

⚠️ **개체 내부 로지스틱은 라벨을 쓴다 — 성능이 아니라 상한**이다(Q7-F/H 의 오라클과
같은 지위). 배포판은 전역 모델에 넣는 **Q7-I′** 다. 여기서 묻는 건 「특징에 값이
있는가」이지 「모델이 얼마 나오는가」가 아니다.

⚠️ **런 우세 층은 `RR` 만으로 0.9905** 다(Q7-F 【F-E】④). **그 층의 개선은 성과가
아니다** — 그래서 I4(고립 S)와 I5(런, 비열등)를 나눠 걸었다.

## 【I-0】 — 외부 리뷰가 지적한 **Q7-F 미결 3건을 먼저 닫는다**

리뷰 지적을 그대로 받되, **단정은 측정으로 대체**한다.

| # | 지적 | 이 노트북의 처리 |
|---|---|---|
| ② | F4 의 `P_late` 대조가 실패했다(rho +0.289, CI 가 0 을 배제). 교란의 정체는 **RR 격차 그 자체** | **편상관** `rho(T중첩, P_early | RR격차)` 를 계산한다. 0 에 붙으면 T 침입은 RR 위에 아무것도 안 더한 것 |
| ③ | F1 의 46개체가 **런 우세 개체를 구조적으로 배제**한다 | 정합 불가 13 ∩ 런 우세 5 를 **교집합으로 출력**한다 |
| ④ | 「RR 격차 중앙 0.320」이 정합 **전/후** 불명확 | 정합 **전·후를 나란히** 낸다 |
| ⑤ | 리듬 라벨 2종은 **파싱 버그**다 (SVDB 에 `(AFIB` 등이 있다) | ⚠️ **단정하지 않는다.** `wfdb.rdann` 으로 **78개체 aux_note 를 직접 전수 파싱**해 npz 와 대조한다. 주석이 있는데 npz 에 없으면 **버그**, 둘 다 없으면 **데이터에 없다** |

### ⚠️ 리뷰 ①(창이 P 파에 안 맞는다)은 **산수를 다시 해야 한다**

리뷰는 「PR 160ms 에서 `P_late` 이 P 파의 **11%** 만 덮는다」고 했다. 그 계산은 P 파를
`R−220 ~ R−120ms` 에 놓는데, 그러면 **P 끝이 R 앞 120ms** 라 PR(P 시작→QRS 시작)이
220ms 가 되어 정상 범위를 벗어난다.

표준 정의로 다시 하면 — PR 160ms · P 폭 100ms · QRS 시작 ≈ R−30ms 이면
**P 는 `R−190 ~ R−90ms`**. 그러면:

| 창 | 범위 | P(R−190~−90) 중 덮는 비율 |
|---|---|---:|
| `P_early` | R−278 ~ −189ms | **~1%** |
| `P_late` | R−131 ~ −42ms | **~41%** |

즉 **방향은 설계대로**다(late 가 P 를 훨씬 많이 덮는다). 다만 **41% 지 100% 가 아니고**,
**PR 은 환자마다·박동종류마다 다르다**(PAC 의 P′ 는 이소성 초점이라 PR 이 다를 수 있다).
→ **리뷰의 결론(개체별 P 정렬 창이 필요하다)은 옳고, 근거 수치는 정정한다.**
그건 **Q7-F′** 로 따로 뗀다 — 이 노트북은 리듬 축이 목적이다. (U-Net 분절기는
이 repo 에 없다. 개체별 **N 템플릿의 P 봉우리**로 정렬하는 게 학습 0회 대안이다.)

## Q7-G 는 **취소가 아니라 보류**로 고친다

Q7-F 노트북의 결론 문장 「전이시킬 형태 신호가 애초에 없다」는 **데이터보다 세다**.
**심박수 정합 후에도 오라클 0.6890 이고 0.5 가 아니다.** 정확한 진술은:

> 형태 신호는 **존재하나**(라벨 쓴 상한 0.6890), Q7-H 의 0.9008 중 **대부분은 심박수
> 차이가 만든 것**이다. 형태 단독 상한은 0.69 이고 배포 모델은 그보다 낮다.

⚠️ 단 그 0.6890 도 **정합이 `pre_rr` 만 통제**한다 — `post_rr`·국소 리듬 문맥은 안
맞췄으므로 **잔여에도 리듬이 남아 있을 수 있다**. 0.69 는 형태의 **상한의 상한**이다.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, glob, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0   = 20260803
IDX_S   = 1
RPRE    = 100
NB_BOOT = 4000
NB_REC  = 300
STT_SEG = (130, 215)   # Q7-F 승계 — 음성 대조 창(P_full 과 동폭)

# ── 사전등록 상수. **이 아래 어느 셀에서도 다시 고르지 않는다.**
KS         = (8, 16)   # 국소 기준 RR 창(비트 수)
MIN_S_TPL  = 20        # Q7-F/H 승계
K_FOLD     = 5         # Q7-H 승계 (R22)
N_REPEAT   = 3         # Q7-H 승계 (겹 배정 잡음을 SE 에 넣는다)
BAND_FRAC  = 0.05      # Q7-F 승계 — 심박수 정합 대역폭
BAND_MIN   = 8
MIN_BAND_S, MIN_BAND_N = 3, 3
MIN_MATCH_S, MIN_MATCH_PAIR = 20, 200
MIN_MATCH_REC = 15
ISO_HI, ISO_LO = 0.7, 0.3   # 층 경계 — Q7-F 【F-E】④ 승계
NI_RUN     = 0.02      # I5 비열등 여유

CONFIG = dict(
    exp="quest46_q7i_rhythm_bundle", quest="ailab-2026-0046", step="svdb-rhythm-bundle",
    parent_exp=["quest46_q7f_window_confound", "ailab-2026-0057"],
    purpose=("형태 축이 리듬 대리로 판정된 뒤, **방치돼 있던 리듬 축**에 특징을 더해 "
             "값어치를 잰다. 핵심은 지금껏 안 쓴 `post_rr`(보상성 휴지)가 **심박수 정합 "
             "조건에서도 남는가**이다"),
    dataset="SVDB 전수 · svdb_data5.npz + Q7-B 예측 캐시(라벨·매핑용)",
    ks=list(KS), stt_seg=list(STT_SEG),
    predictions={
        "I1": "짝지은 LR(전부) − LR(f1) CI 하한 > 0",
        "I2": "짝지은 LR(전부) − STT(창 음성 대조) CI 하한 > 0",
        "I2b": "짝지은 LR(전부) − LR(라벨셔플 null) CI 하한 > 0",
        "I3": "심박수 정합 조건의 f3(보상성 휴지) 매크로 CI 하한 > 0.5",
        "I4": "고립 S 층에서 LR(전부) − LR(f1) CI 하한 > 0",
        "I5": f"런 우세 층에서 LR(전부) − LR(f1) CI 하한 > −{NI_RUN}"},
    caveat=("**개체 내부 로지스틱은 라벨을 쓴다 — 성능이 아니라 상한**이다. 배포판은 "
            "전역 모델에 넣는 Q7-I′. `f1` 도 같은 교차적합 로지스틱에 태워 **공정 비교**한다. "
            "**런 우세 층은 RR 만으로 0.9905**(Q7-F)라 그 층의 개선은 성과가 아니다 — "
            "I4(고립)와 I5(런·비열등)를 나눠 건 이유다. 무정합 조건의 숫자는 Q7-F 가 "
            "**해석 불가**로 확정했다(R24)."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7i_rhythm_bundle", CONFIG, project=PROJECT)
run.log("설정 ✅ 리듬 특징 5종 · 핵심은 f3(post_rr) · 정합 조건이 주지표")

In [ ]:
# CELL 2 — 【G0】 자산 · 매핑 (Q7-D/E/F/H 와 동일 규약 — fallback 없음 R16)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb

P = dict(np.load(PROB))
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
recs = [int(r) for r in wfdb.get_record_list("svdb")]      # ⛔ fallback 없음 (R16)
if len(recs) < 2:
    raise AssetError("SVDB 목록을 못 받았다 — 연속 번호로 대체하지 않는다")
labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError("라벨이 목록 밖인데 연속 가정과도 안 맞는다")
    MAP = {l: recs[l - CONT[0]] for l in labels}
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)
assert len(np.unique(REC)) == len(labels) and not (set(REC.tolist()) - set(recs))

d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"길이 불일치 {int(keep.sum())} vs {len(Y)}"
PRE  = np.asarray(d5["pre_rr"])[keep].astype(float)
POST = np.asarray(d5["post_rr"])[keep].astype(float)      # ★ 처음 쓴다
BSTT = np.ascontiguousarray(np.asarray(d5["beat"])[keep][:, :, STT_SEG[0]:STT_SEG[1]]).astype("float32")
ALLR = [int(r) for r in np.unique(REC)]
run.log("\n" + "=" * 100)
run.log("【G0】 자산 · 매핑")
run.log("=" * 100)
run.log(f"  예측 {len(Y):,}비트 · 레코드 {len(ALLR)}개 · post_rr 적재 {POST.shape} (★ 최초 사용)")
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【I-0】 선행 확인 — 외부 리뷰가 지적한 **Q7-F 미결 3건**을 닫는다
#   ★ 지적을 그대로 받되 **단정은 측정으로 대체**한다. 특히 ⑤(리듬 주석)는
#     "파싱 버그다" 를 가정하지 않고 **원본 주석을 직접 전수 파싱**해서 가른다.
run.log("\n" + "=" * 100)
run.log("【I-0】 선행 확인 (리뷰 지적 ②③④⑤)")
run.log("=" * 100)
PRE0 = {}

# ── ⑤ 리듬 주석 직독 — npz 를 안 거친다
from collections import Counter
def _parse_rhy(a):
    if not a:
        return None
    s = str(a).strip("\x00").strip()
    if not s.startswith("("):
        return None
    return s[1:].strip().strip("\x00") or None

raw_cnt, raw_rec, nonempty = Counter(), {}, 0
for r in recs:
    try:
        ann = wfdb.rdann(str(r), "atr", pn_dir="svdb")
    except Exception as e:
        run.log(f"  ⚠️ #{r} 주석 읽기 실패: {e} — **건너뛰되 기록한다**"); continue
    aux = list(getattr(ann, "aux_note", []) or [])
    for a in aux:
        nm = _parse_rhy(a)
        if nm:
            raw_cnt[nm] += 1; raw_rec.setdefault(nm, set()).add(int(r)); nonempty += 1
run.log(f"\n  ⑤ 원본 주석 직독 — 78개체 전수 · **리듬 표기 {nonempty:,}건**")
if raw_cnt:
    for nm, c in raw_cnt.most_common(12):
        run.log(f"     ({nm:<6} {c:>6,}건 · {len(raw_rec[nm]):>2}개체")
else:
    run.log("     리듬 표기가 **한 건도 없다**")
npz_names = [str(x) for x in np.asarray(d5["rhythm_names"]).tolist()] if "rhythm_names" in set(d5.files) else []
run.log(f"     npz `rhythm_names` = {npz_names}")
_only_raw = sorted(set(raw_cnt) - set(npz_names))
if _only_raw:
    run.log(f"     ⛔ **파싱 버그다** — 원본에는 있는데 npz 에 없는 리듬: {_only_raw}")
    run.log("        → `svdb_labels.rhythm_per_beat` 를 고치고 npz 를 재빌드해야 한다")
elif not raw_cnt:
    run.log("     ✅ **데이터에 없다** — 파싱 버그가 아니다. SVDB 로는 리듬 층화를 못 한다")
    run.log("        → 리듬 층화가 필요하면 mitdb 등 다른 DB 로 간다")
else:
    run.log("     ✅ 원본과 npz 의 리듬 집합이 일치 — 파싱은 정상이다")
PRE0["rhythm_raw"] = {k: int(v) for k, v in raw_cnt.items()}
PRE0["rhythm_npz"] = npz_names
PRE0["rhythm_parse_bug"] = bool(_only_raw)

# ── ②③④ Q7-F 산출물에서 — 없으면 **추측하지 않고 생략**
cfgs = sorted(glob.glob(os.path.join(PROJECT, "runs", "*q7f_window_confound", "config.json")))
if not cfgs:
    run.log("\n  ⚠️ Q7-F config.json 을 못 찾았다 — 지적 ②③④ 확인 **생략**(맞춰 넣지 않는다)")
    QF = None
else:
    QF = json.load(open(cfgs[-1]))
    run.log(f"\n  Q7-F 산출물: {os.path.dirname(cfgs[-1]).split('/')[-1]}")
    QP = {int(k): v for k, v in QF["per_record"].items()}
    qr = sorted(QP)
    ov = np.array([QP[r]["ov_gap_early"] for r in qr])
    pe = np.array([QP[r]["P_early"] for r in qr])
    pl = np.array([QP[r]["P_late"] for r in qr])
    rg = np.array([QP[r]["rr_gap"] for r in qr])

    # ② 편상관 — RR 격차를 통제하면 T 침입이 남는가 (R23 의 처방)
    def partial_spearman(x, y, z):
        x, y, z = (np.asarray(v, float) for v in (x, y, z))
        m = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
        if m.sum() < 6:
            return float("nan")
        rx, ry, rz = (stats.rankdata(v[m]) for v in (x, y, z))
        def resid(a, b):
            b1 = np.c_[np.ones(len(b)), b]
            return a - b1 @ np.linalg.lstsq(b1, a, rcond=None)[0]
        return float(stats.spearmanr(resid(rx, rz), resid(ry, rz)).statistic)
    r_raw = float(stats.spearmanr(ov, pe).statistic)
    r_par = partial_spearman(ov, pe, rg)
    r_par_l = partial_spearman(ov, pl, rg)
    run.log(f"  ② rho(T중첩, P_early) {r_raw:+.3f}  →  **RR격차 통제 편상관 {r_par:+.3f}**")
    run.log(f"     (대조) 같은 편상관을 P_late 에: {r_par_l:+.3f}")
    run.log("     → 편상관이 0 에 붙으면 **T 침입은 RR 격차 위에 아무것도 안 더했다**")
    run.log("       = 교란의 정체는 「직전 T」가 아니라 **「RR 격차 그 자체」**다 (리뷰 ② 지적이 맞다)")
    PRE0["partial_rho"] = dict(raw=r_raw, given_rrgap=r_par, late_given_rrgap=r_par_l)

    # ③ F1 코호트 편향 — 정합 불가 개체와 런 우세 개체의 교집합
    unusable = set(QF.get("match_cost", {}).get("unusable", []))
    runny = {int(k) for k, v in QF.get("diagnostics", {}).get("run_structure", {}).items()
             if v.get("iso_frac", 1.0) < ISO_LO}
    run.log(f"  ③ F1 정합 불가 {len(unusable)}개체 · 런 우세 {len(runny)}개체 · "
            f"**교집합 {sorted(unusable & runny)}**")
    run.log(f"     → F1 의 −0.1915 는 **런 우세 개체를 구조적으로 배제한 코호트**의 값이다")
    PRE0["f1_cohort_bias"] = dict(unusable=sorted(unusable), runny=sorted(runny),
                                  overlap=sorted(unusable & runny))

    # ④ RR 격차 — 정합 **전** 값임을 못박고, 정합 후 잔여는 이 노트북에서 새로 잰다
    run.log(f"  ④ Q7-F 가 출력한 「S/N 중앙 RR 격차 중앙 {np.nanmedian(rg):.3f}」은 **정합 전** 값이다")
    run.log("     (`rr_gap` 은 전 비트로 계산된다). **정합 후 잔여 격차는 【I-B】에서 새로 낸다**")
    PRE0["rr_gap_premeatch_median"] = float(np.nanmedian(rg))
CONFIG["prechecks"] = PRE0
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【I-A】 리듬 특징 5종 + 런 구조
#   ★ 전부 **라벨 불필요**. 레코드 안에서는 비트가 시간순으로 저장돼 있다.
def local_base(pre_v, k):
    """직전 k 비트의 중앙 RR. 앞쪽 가장자리는 **레코드 중앙으로 채우고 표시**한다."""
    n = len(pre_v); out = np.empty(n); med = float(np.median(pre_v))
    edge = np.zeros(n, bool)
    for i in range(n):
        a = max(0, i - k)
        if i - a < 3:
            out[i] = med; edge[i] = True
        else:
            out[i] = float(np.median(pre_v[a:i]))
    return out, edge

def rhythm_feats(pre_v, post_v, ks):
    """f1..f5. 크면 'S 스러움'. f4 는 방향 없는 문맥 변수다."""
    med = float(np.median(pre_v))
    F, NM = [], []
    F.append(med - pre_v);                       NM.append("f1")
    b_first = None
    for k in ks:
        b, _e = local_base(pre_v, k)
        if b_first is None:
            b_first = b
        F.append(1.0 - pre_v / np.maximum(b, 1e-9)); NM.append(f"f2_{k}")
    F.append(1.0 - (pre_v + post_v) / np.maximum(2.0 * b_first, 1e-9)); NM.append("f3")
    cv = np.empty(len(pre_v))
    for i in range(len(pre_v)):
        a = max(0, i - ks[0])
        w = pre_v[a:i] if i - a >= 3 else pre_v[:max(3, 1)]
        cv[i] = float(np.std(w) / max(np.mean(w), 1e-9))
    F.append(cv);                                NM.append("f4")
    f2 = F[1]
    F.append(np.r_[0.0, f2[:-1]]);               NM.append("f5")
    return np.stack(F, axis=1), NM

def run_struct(t_):
    prev_ = np.r_[False, t_[:-1]]; next_ = np.r_[t_[1:], False]
    iso = t_ & ~prev_ & ~next_
    rl = mx = 0
    for v in t_:
        rl = rl + 1 if v else 0
        mx = max(mx, rl)
    return float(iso.sum() / max(t_.sum(), 1)), int(mx)

run.log("\n" + "=" * 100)
run.log("【I-A】 리듬 특징")
run.log("=" * 100)
_p, _n = rhythm_feats(PRE[REC == ALLR[0]], POST[REC == ALLR[0]], KS)
run.log(f"  특징 {len(_n)}종: {_n}  (f4 는 방향 없는 문맥 변수 — 단독 0.5 근처가 정상)")
run.log(f"  ★ f3 만 `post_rr` 을 쓴다 — **심박수 정합 후에도 남을 수 있는 유일한 리듬 특징**")
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【I-B】 전수 채점 — 단독 · 교차적합 조합 · 심박수 정합
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

def dist(B, ref):
    d = B - ref[None]
    return np.sqrt((d * d).sum(axis=(1, 2)))

def cv_logit(X, tt, K, seed, n_rep):
    """되풀이 K겹 교차적합 로지스틱 (R22). 반환 (점수평균, 되풀이간 AUROC SD)."""
    accs, aus = [], []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None, None
            mu = X[tr].mean(0); sd = X[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=2000, C=1.0)
            lr.fit((X[tr] - mu) / sd, tt[tr].astype(int))
            sc[te] = lr.decision_function((X[te] - mu) / sd)
        accs.append(sc); aus.append(roc_auc_score(tt.astype(int), sc))
    return np.mean(np.stack(accs), axis=0), (float(np.std(aus, ddof=1)) if len(aus) > 1 else 0.0)

def two_template_cv(B, tt, K, seed, n_rep):
    accs, aus = [], []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None, None
            medN = np.median(B[tr & ~tt], axis=0); medS = np.median(B[tr & tt], axis=0)
            sc[te] = dist(B[te], medN) - dist(B[te], medS)
        accs.append(sc); aus.append(roc_auc_score(tt.astype(int), sc))
    return np.mean(np.stack(accs), axis=0), (float(np.std(aus, ddof=1)) if len(aus) > 1 else 0.0)

def boot_se(tt, sc, seed, nb):
    rng = np.random.RandomState(seed); v = []
    for _ in range(nb):
        j = rng.randint(0, len(tt), len(tt)); tj = tt[j]
        if 0 < tj.sum() < len(tj):
            v.append(roc_auc_score(tj.astype(int), sc[j]))
    return float(np.std(v, ddof=1)) if len(v) >= 20 else float("nan")

def matched_auc(sc, tt, pre_v, band_w, min_s, min_n):
    """같은 `pre_rr` 대역 안에서만 쌍을 센다. **평가만 제한**한다(Q7-F 승계).
    반환 (조건부 AUROC, 남은 S, 남은 N, 쌍, 대역수, **정합 후 잔여 RR 격차**)."""
    b = np.floor(pre_v / max(band_w, 1e-9)).astype(np.int64)
    num = den = 0.0; ks_ = kn_ = nb_ = 0; ds = []; dn = []
    for bb in np.unique(b):
        m = b == bb
        s_, n_ = sc[m & tt], sc[m & ~tt]
        if len(s_) < min_s or len(n_) < min_n:
            continue
        ks_ += len(s_); kn_ += len(n_); nb_ += 1
        ds.append(pre_v[m & tt]); dn.append(pre_v[m & ~tt])
        gt = float((s_[:, None] > n_[None, :]).sum())
        eq = float((s_[:, None] == n_[None, :]).sum())
        num += gt + 0.5 * eq; den += float(len(s_) * len(n_))
    if den < 1:
        return float("nan"), ks_, kn_, 0.0, nb_, float("nan")
    a_ = np.concatenate(ds); b_ = np.concatenate(dn)
    resid = float(abs(np.median(b_) - np.median(a_)) / max(np.median(pre_v), 1e-9))
    return num / den, ks_, kn_, den, nb_, resid

run.log("\n" + "=" * 100)
run.log(f"【I-B】 전수 채점 ({K_FOLD}겹 교차적합 × {N_REPEAT}회)")
run.log("=" * 100)
PER, SKIP = {}, []
FEAT_ALL = None
for r in ALLR:
    mm = np.where(REC == r)[0]
    tt = (Y[mm] == IDX_S)
    ns, nn = int(tt.sum()), int((~tt).sum())
    if ns < MIN_S_TPL or nn < MIN_S_TPL:
        SKIP.append((int(r), f"S {ns} · N {nn} — 최소 {MIN_S_TPL} 미달")); continue
    pre_m, post_m = PRE[mm], POST[mm]
    X, NAMES = rhythm_feats(pre_m, post_m, KS)
    FEAT_ALL = NAMES
    iso_f, mx_run = run_struct(tt)
    row = dict(n=int(len(mm)), pos=ns, prev=float(tt.mean()), iso_frac=iso_f, max_run=mx_run)
    # 단독 특징
    for j, nm_ in enumerate(NAMES):
        row[nm_] = float(roc_auc_score(tt.astype(int), X[:, j]))
    # 조합 (공정 비교 — f1 도 같은 절차에 태운다)
    ARMS = {"lr_f1": [NAMES.index("f1")],
            "lr_all": list(range(len(NAMES))),
            "lr_norr": [NAMES.index(n) for n in ("f3", "f4") if n in NAMES]}
    SC = {}
    bad = False
    for nm_, cols in ARMS.items():
        sc_, rsd = cv_logit(X[:, cols], tt, K_FOLD, SEED0, N_REPEAT)
        if sc_ is None:
            bad = True; break
        SC[nm_] = sc_
        row[nm_] = float(roc_auc_score(tt.astype(int), sc_))
        se_ = boot_se(tt, sc_, SEED0 + int(r), NB_REC)
        row[nm_ + "_se"] = float(np.sqrt(se_ ** 2 + rsd ** 2)) if np.isfinite(se_) else float("nan")
    if bad:
        SKIP.append((int(r), "겹 안 클래스 부족")); continue
    # ── ★ 라벨 셔플 null — **같은 파이프라인**이 신호 0에서 얼마를 내는지 (픽스처가 잡은 것)
    #    교차적합 로지스틱은 특징 수가 많을수록 null 편의가 커진다. LR(f1)(1개)과
    #    LR(전부)(6개)를 그냥 비교하면 그 편의 차가 곧 결론이 된다.
    rng_p = np.random.RandomState(SEED0 + 777 + int(r))
    tt_p = rng_p.permutation(tt)
    sc_p, _ = cv_logit(X, tt_p, K_FOLD, SEED0, 1)
    row["lr_null"] = float(roc_auc_score(tt_p.astype(int), sc_p)) if sc_p is not None else float("nan")
    # 음성 대조 — Q7-F 의 STT 두 템플릿 (라벨 사용 상한)
    sc_s, rsd_s = two_template_cv(BSTT[mm], tt, K_FOLD, SEED0, N_REPEAT)
    row["stt"] = float(roc_auc_score(tt.astype(int), sc_s)) if sc_s is not None else float("nan")
    SC["stt"] = sc_s if sc_s is not None else np.full(len(tt), np.nan)
    # 심박수 정합 — f1·f2 는 정의상 무력화된다. f3 가 남는지가 관건
    bw = max(BAND_MIN, BAND_FRAC * np.median(pre_m))
    for nm_, vec in (("f1", X[:, NAMES.index("f1")]), ("f3", X[:, NAMES.index("f3")]),
                     ("lr_all", SC["lr_all"]), ("lr_norr", SC["lr_norr"]), ("stt", SC["stt"])):
        if not np.isfinite(vec).all():
            row[nm_ + "_m"] = float("nan"); continue
        a_, ks_, kn_, pr_, nb_, resid = matched_auc(vec, tt, pre_m, bw, MIN_BAND_S, MIN_BAND_N)
        row[nm_ + "_m"] = a_
        if nm_ == "f1":
            row.update(match_s=int(ks_), match_pairs=float(pr_), match_bands=int(nb_),
                       match_s_frac=float(ks_ / max(ns, 1)), rr_gap_post=resid)
    row["rr_gap_pre"] = float(abs(np.median(pre_m[~tt]) - np.median(pre_m[tt]))
                              / max(np.median(pre_m), 1e-9))
    row["match_ok"] = bool(row.get("match_s", 0) >= MIN_MATCH_S
                           and row.get("match_pairs", 0) >= MIN_MATCH_PAIR)
    PER[int(r)] = row

run.log(f"  채점 {len(PER)}개체 · 제외 {len(SKIP)}개체 (전부 아래에)")
for r, why in SKIP:
    run.log(f"    #{r}  {why}")
if len(PER) < 10:
    raise AssetError("채점된 개체가 너무 적다")
RS = sorted(PER)
if FEAT_ALL is None:
    raise AssetError("특징 이름을 못 잡았다 — 모든 개체가 제외됐다")
KEYS = list(FEAT_ALL) + ["lr_f1", "lr_all", "lr_norr", "stt", "lr_null",
                         "f1_m", "f3_m", "lr_all_m", "lr_norr_m", "stt_m",
                         "prev", "iso_frac", "max_run", "match_s_frac",
                         "rr_gap_pre", "rr_gap_post"]
A = {k: np.array([PER[r].get(k, np.nan) for r in RS], float) for k in KEYS}
MOK = np.array([PER[r]["match_ok"] for r in RS], bool)

# ── ④ 정합 전/후 RR 격차를 **나란히** (리뷰 지적)
_pairs = np.array([PER[r].get("match_pairs", np.nan) for r in RS], float)
run.log(f"\n  ★ 심박수 정합 — 가능 개체 **{int(MOK.sum())}/{len(RS)}** · 남은 S 비율 중앙 "
        f"{np.nanmedian(A['match_s_frac']):.3f} · 쌍 수 중앙 {np.nanmedian(_pairs):,.0f}")
run.log(f"    **S/N 중앙 RR 격차 — 정합 전 {np.nanmedian(A['rr_gap_pre']):.3f} → "
        f"정합 후 {np.nanmedian(A['rr_gap_post'][MOK]):.3f}**  (리뷰 지적 ④)")

run.log(f"\n  {'점수':<30}{'매크로':>9}{'SD':>9}{'최소':>9}{'<0.5':>7}{'정합':>9}")
NM2 = {"f1": "f1 med(pre)−pre (기존)", "f2_8": "f2 국소기준 k=8", "f2_16": "f2 국소기준 k=16",
       "f3": "★f3 보상성 휴지(post_rr)", "f4": "f4 국소 불규칙성(문맥)", "f5": "f5 직전 조기성",
       "lr_f1": "LR(f1)  공정 기준선", "lr_all": "★LR(전부)", "lr_norr": "LR(f3,f4) RR 비의존",
       "stt": "STT 음성 대조(창)", "lr_null": "LR(전부)·라벨셔플 null"}
for k in ("f1", "f2_8", "f2_16", "f3", "f4", "f5", "lr_f1", "lr_all", "lr_norr", "stt", "lr_null"):
    if k not in A:
        continue
    v = A[k]; f = np.isfinite(v)
    mk = A.get(k + "_m")
    ms = f"{np.nanmean(mk[MOK]):.4f}" if mk is not None and np.isfinite(mk[MOK]).any() else "—"
    run.log(f"  {NM2.get(k,k):<30}{np.nanmean(v):>9.4f}{np.nanstd(v[f], ddof=1):>9.4f}"
            f"{np.nanmin(v):>9.4f}{int((v[f] < 0.5).sum()):>7}{ms:>9}")
run.log("\n  ⚠️ LR 조합·STT 는 **라벨을 쓰는 상한**이다 — 배포판은 Q7-I′(전역 모델).")
run.log("     f1·f2 는 `pre_rr` 의 함수라 **정합하면 정의상 0.5 로 간다**. f3 만 살아남을 수 있다")
CONFIG["per_record"] = {str(r): PER[r] for r in RS}
CONFIG["skipped"] = [{"rec": r, "why": w} for r, w in SKIP]
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【I-C】 관문 I1·I2·I3
def boot_diff(a, b, seed, nb=NB_BOOT, mask=None, const=None):
    a = np.asarray(a, float)
    d = (a - const) if const is not None else (a - np.asarray(b, float))
    if mask is not None:
        d = d[mask]
    d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.array([d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)])
    return float(d.mean()), float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)), len(d)

run.log("\n" + "=" * 100)
run.log("【I-C】 관문")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

m1, lo1, hi1, n1_ = boot_diff(A["lr_all"], A["lr_f1"], SEED0 + 1)
DIFF["I1"] = dict(mean=m1, lo=lo1, hi=hi1, n=n1_)
g_("I1", decide(lo1, hi1, 0.0, ">"),
   f"짝지은 LR(전부) − LR(f1) **{m1:+.4f}** [{lo1:+.4f}, {hi1:+.4f}] · n={n1_}"
   f" · 좋아진 개체 {int(np.nansum(A['lr_all'] > A['lr_f1']))}/{n1_}")

m2, lo2, hi2, n2_ = boot_diff(A["lr_all"], A["stt"], SEED0 + 2)
DIFF["I2"] = dict(mean=m2, lo=lo2, hi=hi2, n=n2_)
g_("I2", decide(lo2, hi2, 0.0, ">"),
   f"짝지은 LR(전부) − STT(음성 대조) **{m2:+.4f}** [{lo2:+.4f}, {hi2:+.4f}] · n={n2_}"
   f"   ← 최소 관문. 못 이기면 특징이 창 잡음만도 못한 것")

m2b, lo2b, hi2b, n2b = boot_diff(A["lr_all"], A["lr_null"], SEED0 + 6)
DIFF["I2b"] = dict(mean=m2b, lo=lo2b, hi=hi2b, n=n2b)
g_("I2b", decide(lo2b, hi2b, 0.0, ">"),
   f"짝지은 LR(전부) − LR(**라벨셔플**) **{m2b:+.4f}** [{lo2b:+.4f}, {hi2b:+.4f}] · n={n2b}"
   f"   ← 같은 파이프라인의 null 수준. 특징 수가 다르면 LR(f1) 비교만으론 부족하다")
run.log(f"    라벨셔플 null 매크로 {np.nanmean(A['lr_null']):.4f} "
        f"(0.5 에서 멀면 교차적합 편의가 크다는 뜻 — I1 을 그만큼 할인해 읽는다)")

# ── I3 ★ 정합 조건에서 f3 가 0.5 를 넘는가 — 이 실험의 핵심
if int(MOK.sum()) < MIN_MATCH_REC:
    VERD["I3"] = "⚠️ 미결"
    DIFF["I3"] = dict(n=int(MOK.sum()), reason="정합 가능 개체 부족")
    run.log(f"  I3  ⚠️ 미결  정합 가능 개체 {int(MOK.sum())} < {MIN_MATCH_REC} — 조용히 판정하지 않는다")
else:
    m3, lo3, hi3, n3_ = boot_diff(A["f3_m"], None, SEED0 + 3, mask=MOK, const=0.5)
    DIFF["I3"] = dict(mean=m3 + 0.5, lo=lo3 + 0.5, hi=hi3 + 0.5, n=n3_)
    g_("I3", decide(lo3, hi3, 0.0, ">"),
       f"정합 조건 f3(보상성 휴지) 매크로 **{m3+0.5:.4f}** [{lo3+0.5:.4f}, {hi3+0.5:.4f}]"
       f" · n={n3_} vs 우연 0.5")
    for k_, tag in (("f1_m", "f1 (정합하면 무력화되어야 정상)"), ("lr_norr_m", "LR(f3,f4)"),
                    ("lr_all_m", "LR(전부)"), ("stt_m", "STT 음성 대조")):
        mm_, lo_, hi_, _ = boot_diff(A[k_], None, SEED0 + 4, mask=MOK, const=0.5)
        run.log(f"    (참고) 정합 {tag:<32} {mm_+0.5:.4f} [{lo_+0.5:.4f}, {hi_+0.5:.4f}]")

# ── 최대 기여 개체 (R11-c)
d1 = A["lr_all"] - A["lr_f1"]; fin = np.isfinite(d1)
tot = float(np.abs(d1[fin]).sum())
top = sorted(np.where(fin)[0], key=lambda i: -abs(d1[i]))[:5]
run.log(f"\n  I1 기여 상위 5 (|Δ| 합 {tot:.3f} 중):")
for i in top:
    run.log(f"    #{RS[i]}  Δ {d1[i]:+.4f} ({abs(d1[i])/max(tot,1e-9)*100:.1f}%)"
            f" · 유병률 {A['prev'][i]:.3f} · 고립S {A['iso_frac'][i]:.2f} · 최장런 {int(A['max_run'][i])}")
if top:
    mb_, lb_, hb_, _ = boot_diff(np.delete(A["lr_all"], top[0]), np.delete(A["lr_f1"], top[0]), SEED0 + 5)
    run.log(f"  최대 기여 개체 #{RS[top[0]]} 제외 → **{mb_:+.4f}** [{lb_:+.4f}, {hb_:+.4f}]")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【I-D】 층별 관문 I4·I5 + 특징 제거 실험
#   ★ Q7-F 【F-E】④ 가 층을 실측으로 만들었다 — **런 우세 층은 RR 만으로 0.9905** 다.
#     거기서의 개선은 성과가 아니므로 I4(고립)와 I5(런·비열등)를 나눠 건다.
ISO = A["iso_frac"]
S_ISO = ISO >= ISO_HI; S_MIX = (ISO >= ISO_LO) & (ISO < ISO_HI); S_RUN = ISO < ISO_LO
run.log("\n" + "=" * 100)
run.log("【I-D】 층별 · 제거 실험")
run.log("=" * 100)
run.log(f"  층 크기 — 고립 S 우세 {int(S_ISO.sum())} · 혼합 {int(S_MIX.sum())} · 런 우세 {int(S_RUN.sum())}")

if int(S_ISO.sum()) < 3:
    VERD["I4"] = "⚠️ 미결"
    run.log(f"  I4  ⚠️ 미결  고립 S 개체 {int(S_ISO.sum())} < 3 — 판정하지 않는다")
else:
    m4, lo4, hi4, n4_ = boot_diff(A["lr_all"], A["lr_f1"], SEED0 + 11, mask=S_ISO)
    DIFF["I4"] = dict(mean=m4, lo=lo4, hi=hi4, n=n4_)
    g_("I4", decide(lo4, hi4, 0.0, ">"),
       f"**고립 S 층**({n4_}개체) LR(전부) − LR(f1) **{m4:+.4f}** [{lo4:+.4f}, {hi4:+.4f}]")

if int(S_RUN.sum()) < 3:
    VERD["I5"] = "⚠️ 미결"
    run.log(f"  I5  ⚠️ 미결  런 우세 개체 {int(S_RUN.sum())} < 3 — 판정하지 않는다")
else:
    m5, lo5, hi5, n5_ = boot_diff(A["lr_all"], A["lr_f1"], SEED0 + 12, mask=S_RUN)
    DIFF["I5"] = dict(mean=m5, lo=lo5, hi=hi5, n=n5_)
    g_("I5", decide(lo5, hi5, -NI_RUN, ">"),
       f"**런 우세 층**({n5_}개체) LR(전부) − LR(f1) **{m5:+.4f}** [{lo5:+.4f}, {hi5:+.4f}]"
       f" vs 여유 −{NI_RUN}  (해치지 않는가)")

run.log("\n  층별 매크로")
for nm_, msk in (("고립 S 우세", S_ISO), ("혼합", S_MIX), ("런 우세", S_RUN)):
    if msk.sum() < 2:
        continue
    run.log(f"    {nm_:<10}{int(msk.sum()):>3}개체 · f1 {np.nanmean(A['f1'][msk]):.4f}"
            f" · f3 {np.nanmean(A['f3'][msk]):.4f}"
            f" · LR(f1) {np.nanmean(A['lr_f1'][msk]):.4f}"
            f" · **LR(전부) {np.nanmean(A['lr_all'][msk]):.4f}**"
            f" · STT {np.nanmean(A['stt'][msk]):.4f}")

run.log("\n  특징 제거 실험 (단독 AUROC · 기여 분해의 1차 근사)")
for k in ("f1", "f2_8", "f2_16", "f3", "f4", "f5"):
    if k in A:
        run.log(f"    {k:<7} 단독 {np.nanmean(A[k]):.4f}  ·  정합 "
                + (f"{np.nanmean(A[k+'_m'][MOK]):.4f}" if (k + "_m") in A else "—"))
run.log("  ⚠️ 단독 AUROC 는 **상관된 특징끼리 중복**을 센다 — 진짜 기여 분해는 "
        "빼고 다시 적합해야 한다(Q7-I′ 에서 모델과 함께).")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF
CONFIG["strata"] = dict(iso=int(S_ISO.sum()), mix=int(S_MIX.sum()), run=int(S_RUN.sum()))
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 그림 + 마무리
#  ★ Colab 기본 폰트에 한글이 없어 □ 로 깨진다. 그림 라벨은 ASCII 로 쓴다.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
rs_ = np.array(RS)

ax[0].scatter(A["lr_f1"], A["lr_all"], s=34, alpha=.85,
              c=np.where(S_RUN, "crimson", np.where(S_MIX, "goldenrod", "C0")))
ax[0].plot([0, 1], [0, 1], color="gray", ls="--", lw=1)
ax[0].axhline(.5, color="crimson", ls=":", lw=1); ax[0].axvline(.5, color="crimson", ls=":", lw=1)
ax[0].set_xlabel("AUROC, LR(f1) baseline")
ax[0].set_ylabel("AUROC, LR(all rhythm feats)")
ax[0].set_title(f"(1) does the bundle help?  (n={len(rs_)}, red=run-dominant)")
ax[0].grid(alpha=.3); ax[0].set_xlim(-.02, 1.02); ax[0].set_ylim(-.02, 1.02)

KS2 = ["f1", "f2_8", "f3", "f4", "f5", "lr_f1", "lr_all", "stt"]
LB = ["f1\nmed-pre", "f2 k=8\nlocal base", "f3\ncomp. pause", "f4\nlocal CV",
      "f5\nprev early", "LR(f1)", "LR(all)", "STT\n(neg ctrl)"]
CL = ["C7", "C7", "C2", "C7", "C7", "C0", "C0", "C1"]
ax[1].bar(range(len(KS2)), [float(np.nanmean(A[k])) for k in KS2], color=CL)
mm_ = [float(np.nanmean(A[k + "_m"][MOK])) if (k + "_m") in A else np.nan for k in KS2]
ax[1].scatter(range(len(KS2)), mm_, marker="_", s=700, color="k", zorder=5,
              label="rate-matched")
ax[1].axhline(0.5, color="crimson", ls="--", lw=1)
ax[1].set_xticks(range(len(KS2))); ax[1].set_xticklabels(LB, fontsize=7)
ax[1].set_ylabel("macro AUROC")
ax[1].set_title("(2) per-feature   black tick = rate-matched (f1/f2 must collapse)")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3, axis="y"); ax[1].set_ylim(0, 1.05)

ax[2].scatter(A["iso_frac"], A["lr_all"] - A["lr_f1"], s=34, alpha=.85, c=A["prev"], cmap="viridis")
ax[2].axhline(0, color="gray", ls="--", lw=1)
ax[2].axvline(ISO_LO, color="crimson", ls=":", lw=1); ax[2].axvline(ISO_HI, color="crimson", ls=":", lw=1)
cb = plt.colorbar(ax[2].collections[0], ax=ax[2]); cb.set_label("S prevalence", fontsize=8)
ax[2].set_xlabel("isolated-S fraction  (low = run-dominant)")
ax[2].set_ylabel("delta AUROC  LR(all) - LR(f1)")
ax[2].set_title("(3) where does the bundle help?")
ax[2].grid(alpha=.3)
plt.tight_layout(); run.save_fig("q7i_rhythm_bundle", fig); plt.show()

run.log("\n" + "=" * 100)
run.log("관문 요약")
run.log("=" * 100)
for k in ("I1", "I2", "I2b", "I3", "I4", "I5"):
    run.log(f"  {k:<4}{VERD.get(k, '(미실행)')}")
ok = lambda k: VERD.get(k, "").startswith("✅")
if not ok("I2b"):
    run.log("\n  ⛔ **I2b 가 지지가 아니다 — 여기서 멈춘다.** 같은 파이프라인의 라벨셔플 null 을")
    run.log("     못 이겼다는 뜻이라, I1·I4 의 양수는 **교차적합 편의**일 수 있다.")
    run.log("     (합성 null 코호트 실측: LR(f1) 0.394 vs LR(전부) 0.452 — 특징 수만으로 벌어진다)")
if ok("I3"):
    run.log("\n  ★ **`post_rr` 이 심박수 정합 후에도 남는다** — 조기성과 독립인 리듬 정보가 있다.")
    run.log("     Q7-I′(전역 모델 투입)로 간다. 보상성 휴지는 S/V 경계에도 쓰인다")
else:
    run.log(f"\n  ⚠️ I3 {VERD.get('I3')} — 정합 후 남는 리듬 정보를 확인하지 못했다.")
    run.log("     리듬 축도 사실상 「조기성」 하나였다는 뜻이 된다 — 그러면 남은 건 형태뿐이고,")
    run.log("     Q7-F′(P 중심 정렬 창)의 우선순위가 올라간다")
if not ok("I2"):
    run.log(f"  ⛔ I2 {VERD.get('I2')} — **음성 대조를 못 이겼다.** 다른 관문을 읽기 전에 이걸 먼저 본다")
run.finish({"verdicts": VERD, "diffs": DIFF,
            "macro": {k: float(np.nanmean(A[k])) for k in KS2},
            "n_scored": len(RS), "n_match_ok": int(MOK.sum()),
            "prechecks": CONFIG.get("prechecks", {})})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-rhythm-bundle`")